# Task 15 - Dynamic Product Sales Segmentation

**Goal:** use learned boundaries to segment products by sales volume and compare ratings across segments.

**Input:** `data/processed/womens_clothing_final.csv`
**Outputs:** product segment features and reports in `data/processed/` and `reports/`.

> The available product catalog has no customer age column. This notebook segments on `units_sold` rather than fabricating customer age groups.

In [1]:
from pathlib import Path
from datetime import datetime
import re
import numpy as np
import pandas as pd

def find_project_root():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "data" / "processed").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate a project folder containing data/processed.")

ROOT = find_project_root()
PROC = ROOT / "data" / "processed"
REPORTS = ROOT / "reports"
REPORTS.mkdir(parents=True, exist_ok=True)
DATA_PATH = PROC / "womens_clothing_final.csv"
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Expected product catalog CSV at {DATA_PATH}")
df = pd.read_csv(DATA_PATH)
df = df.loc[:, ~df.columns.astype(str).str.startswith("Unnamed")].copy()
print(f"Project: {ROOT}\nLoaded: {DATA_PATH.name} | rows={len(df):,} | columns={len(df.columns)}")

def require_columns(frame, columns):
    missing = [column for column in columns if column not in frame.columns]
    if missing:
        raise KeyError(f"{DATA_PATH.name} is not the expected product-catalog dataset. "
                       f"Missing columns: {missing}. Available columns: {list(frame.columns)}")

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.tree import DecisionTreeClassifier

require_columns(df, ["units_sold", "rating"])
df["units_sold"] = pd.to_numeric(df["units_sold"], errors="coerce")
df["rating"] = pd.to_numeric(df["rating"], errors="coerce")
valid = df["units_sold"].notna() & df["rating"].notna() & (df["units_sold"] >= 0)
sales = df.loc[valid, "units_sold"].astype(float)
log_sales = np.log1p(sales).to_numpy().reshape(-1, 1)
if len(sales) < 10 or sales.nunique() < 3:
    raise ValueError("Need at least 10 valid products and 3 distinct sales values to segment.")
rating_target = (df.loc[valid, "rating"] >= 4).astype(int)
print(f"Products available for segmentation: {len(sales):,}")

Project: D:\OneDrive\Desktop\Brainybeams\Womens_Clothing_ML_Project
Loaded: womens_clothing_final.csv | rows=1,539 | columns=40
Products available for segmentation: 1,539


In [2]:
scan = []
for k in range(2, min(7, len(sales), sales.nunique())):
    labels = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(log_sales)
    score = silhouette_score(log_sales, labels)
    scan.append({"k": k, "silhouette": score})
scan = pd.DataFrame(scan)
best_k = int(scan.loc[scan["silhouette"].idxmax(), "k"])
km = KMeans(n_clusters=best_k, n_init=10, random_state=42).fit(log_sales)
centers = np.sort(km.cluster_centers_.ravel())
km_edges = (centers[:-1] + centers[1:]) / 2
km_segment = pd.cut(log_sales.ravel(), [-np.inf, *km_edges, np.inf], labels=False)
print(f"Selected K-means k={best_k}; silhouette={scan['silhouette'].max():.3f}")
display(scan)

tree = DecisionTreeClassifier(max_leaf_nodes=best_k, min_samples_leaf=max(1, int(0.05 * len(sales))),
                              random_state=42).fit(log_sales, rating_target)
tree_edges = np.sort(np.unique(tree.tree_.threshold[tree.tree_.feature >= 0]))
tree_segment = pd.cut(log_sales.ravel(), [-np.inf, *tree_edges, np.inf], labels=False)

Selected K-means k=6; silhouette=0.968


,k,silhouette
0,2,0.752022
1,3,0.859623
2,4,0.896763
3,5,0.942409
4,6,0.967729


In [3]:
def segment_labels(segment_ids, values):
    result = {}
    for segment_id in sorted(pd.Series(segment_ids).dropna().unique()):
        group_values = values[np.asarray(segment_ids) == segment_id]
        result[int(segment_id)] = f"{int(group_values.min()):,}-{int(group_values.max()):,} units"
    return result

km_labels = segment_labels(km_segment, sales.to_numpy())
tree_labels = segment_labels(tree_segment, sales.to_numpy())
out = df.copy()
out["product_segment_id"] = -1
out["product_segment"] = "Unknown"
out.loc[valid, "product_segment_id"] = km_segment.astype(int)
out.loc[valid, "product_segment"] = pd.Series(km_segment, index=sales.index).map(km_labels).to_numpy()
out["product_segment_tree_id"] = -1
out["product_segment_tree"] = "Unknown"
out.loc[valid, "product_segment_tree_id"] = tree_segment.astype(int)
out.loc[valid, "product_segment_tree"] = pd.Series(tree_segment, index=sales.index).map(tree_labels).to_numpy()
out["product_segment_method"] = "K-means on log1p(units_sold)"

def describe_segments(segment_ids, method):
    labels = np.asarray(segment_ids, dtype=int)
    shares = pd.Series(labels).value_counts(normalize=True)
    rating_values = df.loc[valid, "rating"].to_numpy()
    grand_mean = rating_values.mean()
    total_ss = ((rating_values - grand_mean) ** 2).sum()
    between_ss = sum(
        np.sum(labels == label) * (rating_values[labels == label].mean() - grand_mean) ** 2
        for label in np.unique(labels)
    )
    return {
        "method": method,
        "n_segments": len(shares),
        "silhouette": silhouette_score(log_sales, labels) if len(shares) > 1 else np.nan,
        "rating_eta_squared": between_ss / total_ss if total_ss else np.nan,
        "min_segment_share": shares.min(),
        "balance_entropy": float(-(shares * np.log(shares)).sum() / np.log(len(shares)))
        if len(shares) > 1 else 0.0,
    }

comparison = pd.DataFrame([
    describe_segments(km_segment, "K-means (dynamic sales boundaries)"),
    describe_segments(tree_segment, "Decision tree (rating-guided sales boundaries)"),
])
display(comparison.round(4))

profile = out.loc[valid].groupby(["product_segment_id", "product_segment"]).agg(
    products=("units_sold", "size"),
    mean_units_sold=("units_sold", "mean"),
    median_units_sold=("units_sold", "median"),
    mean_rating=("rating", "mean"),
    high_rating_rate=("rating", lambda values: (values >= 4).mean()),
).reset_index()
if "price" in out.columns:
    profile["mean_price"] = out.loc[valid].groupby("product_segment_id")["price"].mean().values
profile["share"] = profile["products"] / profile["products"].sum()
display(profile.round(3))

out.to_csv(PROC / "womens_clothing_product_segmented.csv", index=False)
comparison.to_csv(REPORTS / "task_15_sales_cluster_comparison.csv", index=False)
profile.to_csv(REPORTS / "task_15_product_segment_summary.csv", index=False)
pd.DataFrame([{
    "task": "Task 15 - Dynamic Product Sales Segmentation", "status": "completed",
    "run_at": datetime.now().isoformat(timespec="seconds"), "input_file": DATA_PATH.name,
    "rows_in": len(df), "rows_segmented": int(valid.sum()), "selected_method": "K-means",
    "n_segments": best_k, "kmeans_log1p_edges": str(km_edges.tolist()),
    "tree_log1p_edges": str(tree_edges.tolist()),
    "output_dataset": "womens_clothing_product_segmented.csv",
}]).to_csv(REPORTS / "task_15_analysis_status.csv", index=False)

,method,n_segments,silhouette,rating_eta_squared,min_segment_share,balance_entropy
0,K-means (dynamic sales boundaries),6,0.9677,0.0567,0.0370,0.8766
1,Decision tree (rating-guided sales boundaries),6,0.9329,0.0411,0.0812,0.9197


,product_segment_id,product_segment,products,mean_units_sold,median_units_sold,mean_rating,high_rating_rate,mean_price,share
0,0,1-10 units,57,8.719,10.0,4.356,0.754,8.566,0.037
1,1,50-50 units,68,50.000,50.0,3.907,0.574,7.830,0.044
2,2,100-100 units,493,100.000,100.0,3.727,0.337,8.058,0.320
3,3,"1,000-1,000 units",403,1000.000,1000.0,3.806,0.345,8.567,0.262
4,4,"5,000-5,000 units",216,5000.000,5000.0,3.819,0.375,8.455,0.140
5,5,"10,000-12,350 units",302,10980.464,10000.0,3.886,0.384,8.314,0.196
